# Bài tập 2 - Huỳnh Nam

Môn 121036. Người phụ trách: **Huỳnh Nam**.

Hoàn thành nội bộ: **18/11/2026**. Hạn nộp: **20/11/2026**.

Bộ khung chưa có thuật toán hoặc kết quả thực nghiệm. Đọc `README.md`, đề Bài tập 2 và bảng phân công của nhóm.

## 0. Chuẩn bị môi trường

Cell bên dưới do Codex tạo để import thư viện và xác định đường dẫn, ngoài phần cài đặt thuật toán. Các cell cài đặt được để trống.

**Khai báo AI cho cell chuẩn bị**

- Công cụ: Codex (OpenAI).
- Mục đích: tạo bộ khung thư mục, notebook và đường dẫn dùng chung.
- Prompt nguyên văn:

> giờ bạn tạo cái sườn giúp tôi [https://github.com/NguyenTanHiep07/xla-121036](https://github.com/NguyenTanHiep07/xla-121036) làm tiếp trong này cho dễ nha tạo mấy thư mục mấy cũ ko liên quan á bỏ qua

In [1]:
from pathlib import Path
import math
import time

import numpy as np
import matplotlib.pyplot as plt

# Chuẩn bị đường dẫn, không cài đặt thuật toán.
thu_muc_chay = Path.cwd().resolve()
thu_muc_tim = (
    thu_muc_chay,
    thu_muc_chay / "bai_tap_2",
    thu_muc_chay.parent,
)
BT2_DIR = next(
    (p for p in thu_muc_tim if (p / "bt2_nhom.ipynb").is_file()),
    None,
)
if BT2_DIR is None:
    raise FileNotFoundError(
        "Chọn thư mục chạy là gốc repo, bai_tap_2 hoặc bai_tap_2/lam_viec."
    )
IMAGES_DIR = BT2_DIR.parent / "images"
print("Bài tập 2:", BT2_DIR)
print("Ảnh dùng chung:", IMAGES_DIR)


Bài tập 2: D:\xla-121036\bai_tap_2
Ảnh dùng chung: D:\xla-121036\images


## Hàm dùng chung từ Gấm

Sau khi bản Harris dùng chung đã được duyệt, thành viên lấy định nghĩa cần thiết vào notebook cá nhân và ghi nguồn/commit. Cell dưới đây dành cho mã tự viết đã được duyệt của nhóm; hiện để trống. Không chạy toàn bộ notebook khác để lấy biến.

# Nguồn: bai_tap_2/lam_viec/bt2_gam.ipynb (Hồng Gấm)

In [2]:
def gaussian_blur(img, sigma):
    img = np.asarray(img, dtype=np.float64)

    if sigma <= 0:
        return img.copy()

    radius = int(np.ceil(3 * sigma))
    x = np.arange(-radius, radius + 1, dtype=np.float64)

    kernel_1d = np.exp(-(x ** 2) / (2 * sigma ** 2))
    kernel_1d /= kernel_1d.sum()

    kernel = np.outer(kernel_1d, kernel_1d)

    padded = np.pad(img, radius, mode="reflect")
    result = np.zeros_like(img)

    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            result += kernel[i, j] * padded[
                i:i + img.shape[0],
                j:j + img.shape[1]
            ]

    return result
def structure_tensor(img, sigma_i, sigma_d):
    img = np.asarray(img, dtype=np.float64)

    if img.ndim != 2:
        raise ValueError("img phải là ảnh xám 2 chiều (H, W)")

    if sigma_i <= 0 or sigma_d <= 0:
        raise ValueError("sigma_i và sigma_d phải dương")

    blurred = gaussian_blur(img, sigma_d)

    Iy, Ix = np.gradient(blurred)

    factor = sigma_d ** 2

    Mxx = factor * gaussian_blur(Ix ** 2, sigma_i)
    Mxy = factor * gaussian_blur(Ix * Iy, sigma_i)
    Myy = factor * gaussian_blur(Iy ** 2, sigma_i)

    return Mxx, Mxy, Myy
def harris_response(img, sigma_i, sigma_d, k=0.04):
    Mxx, Mxy, Myy = structure_tensor(
        img, sigma_i, sigma_d
    )

    det = Mxx * Myy - Mxy ** 2
    trace = Mxx + Myy

    R = det - k * trace ** 2

    return R

def nms_2d(response, window=3):
    response = np.asarray(response, dtype=np.float64)

    if response.ndim != 2:
        raise ValueError("response phải là mảng 2 chiều")

    if window < 1 or window % 2 == 0:
        raise ValueError("window phải là số nguyên dương lẻ")

    radius = window // 2

    padded = np.pad(
        response,
        radius,
        mode="constant",
        constant_values=-np.inf
    )

    mask = np.isfinite(response)

    for di in range(-radius, radius + 1):
        for dj in range(-radius, radius + 1):
            if di == 0 and dj == 0:
                continue

            neighbor = padded[
                radius + di:radius + di + response.shape[0],
                radius + dj:radius + dj + response.shape[1]
            ]

            if di < 0 or (di == 0 and dj < 0):
                mask &= response > neighbor
            else:
                mask &= response >= neighbor

    return mask

def harris_classic(
    img,
    sigma_i=2.0,
    sigma_d=1.4,
    k=0.04,
    thresh_rel=0.01
):
    R = harris_response(img, sigma_i, sigma_d, k)

    max_response = np.max(R)

    if max_response <= 0:
        return np.empty((0, 3), dtype=np.float64)

    mask = nms_2d(R, window=3)
    mask &= R > thresh_rel * max_response
    mask &= R > 0

    rows, cols = np.where(mask)
    responses = R[rows, cols]

    points = np.column_stack((rows, cols, responses))

    order = np.argsort(-points[:, 2], kind="stable")

    return points[order]

def repeatability(pts_a, pts_b, transform, img_shape, eps=3.0):
    pts_a = np.asarray(pts_a, dtype=np.float64)
    pts_b = np.asarray(pts_b, dtype=np.float64)

    if pts_a.size == 0 or pts_b.size == 0:
        return 0.0

    coords_a = pts_a.reshape(-1, pts_a.shape[-1])[:, :2]
    coords_b = pts_b.reshape(-1, pts_b.shape[-1])[:, :2]

    if callable(transform):
        projected = np.array(
            [transform(p) for p in coords_a],
            dtype=np.float64
        ).reshape(-1, 2)
    else:
        H = np.asarray(transform, dtype=np.float64)

        if H.shape != (3, 3):
            raise ValueError("transform phải là ma trận 3x3")

        xy1 = np.column_stack((
            coords_a[:, 1],
            coords_a[:, 0],
            np.ones(len(coords_a))
        ))

        mapped = (H @ xy1.T).T

        projected = np.full(
            (len(coords_a), 2),
            np.nan,
            dtype=np.float64
        )

        valid_w = (
            np.isfinite(mapped[:, 2]) &
            (np.abs(mapped[:, 2]) > 1e-12)
        )

        projected[valid_w, 0] = (
            mapped[valid_w, 1] / mapped[valid_w, 2]
        )

        projected[valid_w, 1] = (
            mapped[valid_w, 0] / mapped[valid_w, 2]
        )

    h, w = img_shape[:2]

    valid = (
        np.isfinite(projected).all(axis=1) &
        (projected[:, 0] >= 0) &
        (projected[:, 0] < h) &
        (projected[:, 1] >= 0) &
        (projected[:, 1] < w)
    )

    projected = projected[valid]

    coords_b = coords_b[
        np.isfinite(coords_b).all(axis=1)
    ]

    if len(projected) == 0 or len(coords_b) == 0:
        return 0.0

    distances = np.linalg.norm(
        projected[:, None, :] -
        coords_b[None, :, :],
        axis=2
    )

    matches = np.min(distances, axis=1) <= eps

    return float(np.mean(matches))


## Cài đặt - KHÔNG dùng AI

Tự viết các hàm ở những cell trống bên dưới. Chữ ký và đầu ra trong markdown là bản nhắc đặc tả của đề; không có mã thuật toán được tạo sẵn. Không dùng AI để viết, sửa lỗi hay viết lại phần này theo quy định của đề.

### `scale_pyramid(sigma0=1.2, xi=1.2, n_levels=10)`

Trả mảng 1D các sigma theo đặc tả.

In [83]:
def scale_pyramid(sigma0=1.2, xi=1.2, n_levels=10):
    sp = [sigma0]
    for n in range(1,n_levels):
        sn = sp[0]*xi**n
        sp.append(sn)
    return np.array(sp)

### `log_normalized(img, sigma)`

Trả độ lớn Laplacian chuẩn hóa theo tỉ lệ, sigma**2 * |Lxx + Lyy|.

In [72]:
def log_normalized(img, sigma):
    L = gaussian_blur(img,sigma)
    P = np.pad(L,1, mode="reflect")
    Lxx=[]
    Lyy=[]
    r,c=L.shape
    for x in range(1,r+1):
        Lxxa=[]
        Lyya=[]
        for n in range(0,c):
            xx = P[x,n]-2*P[x,n+1]+P[x,n+2]
            yy = P[x-1,n+1]-2*P[x,n+1]+P[x+1,n+1]
            Lxxa.append(xx)
            Lyya.append(yy)
        Lxx.append(Lxxa)
        Lyy.append(Lyya)
    return sigma**2 * np.abs(np.array(Lxx)+np.array(Lyy)) 

### `harris_laplace(img, sigmas, k=0.04, thresh_rel=0.01, log_thresh=0.0)`

Trả mảng (N, 4): (row, col, sigma, response). Giữ cực đại LoG theo trục tỉ lệ theo đề.

In [84]:
def harris_laplace(img, sigmas, k=0.04, thresh_rel=0.01, log_thresh=0.0):
    Log = [log_normalized(img,sigma) for sigma in sigmas]
    Result = []
    for n in range(len(sigmas)-1):
        R = harris_response(img,sigmas[n],0.7*sigmas[n],k)
        max_response = np.max(R)
    
        if max_response <= 0:
            continue
    
        mask = nms_2d(R, window=3)
        mask &= R > thresh_rel * max_response
        mask &= R > 0
    
        row, col = np.where(mask)
        if n==0 or n==len(sigmas)-1:
            continue
        for r,c in zip(row,col):
            M = Log[n][r,c]
            F = Log[n-1][r,c]
            B = Log[n+1][r,c]
            if M>F and M>B and M>log_thresh:
                A = r,c,sigmas[n],R[r,c]
                Result.append(A)
    if len(Result)==0:
        return np.array([]).reshape(0,4)
    Re = np.array(Result)
    order = np.argsort(-Re[:, 3], kind="stable")
    return Re[order]

## Kiểm chứng cài đặt

Mã A1-A5 và B1-B5 bên dưới là thứ tự kiểm chứng trong mục 3.7 / 4.5 của đề. Điền dữ liệu kiểm tra, số đo và output thực tế. Các ô mã hiện để trống.

### Kiểm chứng A3 - Phóng to 3 lần

Báo repeatability Harris cổ điển và Harris-Laplace. Người phụ trách: Nam.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét kiểm chứng:**

- Kết quả thực tế: 
- Đạt/chưa đạt và lý do: 

## Thí nghiệm - A1: Harris-Laplace so với Harris cổ điển

Tự chọn dữ liệu, trục quét, số đo và kết luận. Theo mục 5.1, mỗi cặp cần bằng chứng khác biệt, quét tham số có ý nghĩa và giải thích cơ chế. Nếu thực tế dùng AI hỗ trợ cell thí nghiệm, ghi prompt nguyên văn ngay phía trên cell đó.

### Thiết kế và bằng chứng khác biệt

Nêu ảnh nào, vì sao chọn, giữ cố định điều kiện gì và đo đại lượng nào. Trình bày kết quả của hai phương pháp.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



### Quét tham số

Ghi trục quét, giá trị thử và các điều kiện giữ cố định; trình bày đường cong hoặc điểm chuyển hành vi.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



### Giải thích kết quả

Điền kết quả đo, giải thích đại lượng nào thay đổi, ở bước nào và vì sao. Báo cả trường hợp chưa cải thiện.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



## Bàn giao cho nhóm trưởng

- [ ] Đúng giao diện hàm và quy ước tọa độ.
- [ ] Restart & Run All chạy được, đã lưu output.
- [ ] Ảnh cần dùng có trong images/ tại gốc repo; dùng IMAGES_DIR để đọc ảnh.
- [ ] Kiểm chứng và thí nghiệm có số đo thực tế.
- [ ] Cell thí nghiệm có dùng AI ghi đúng prompt ngay phía trên.
- [ ] Đã mở Pull Request và ghi nguồn các hàm dùng chung.